In [1]:
import sys
import os
import math
import numpy as np
import csv
from datetime import datetime

# [UPDATE]: Thêm parent directory vào sys.path để import module từ root project của tác giả
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
print(f"Added to sys.path: {parent_dir}")

# Khai báo thư viện Machine Learning
from sklearn.svm import SVC
from sklearn.neighbors import NearestNeighbors
from sklearn import metrics
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, f1_score
from imblearn.over_sampling import BorderlineSMOTE # [NEW] Thêm SMOTE cho Kịch bản 2

# Import các module lõi từ project gốc của tác giả (KHÔNG ĐƯỢC SỬA CÁC MODULE NÀY)
from wsvm.application import Wsvm
from svm.application import Svm
from fuzzy.weight import fuzzy

Added to sys.path: /home/quangvd/project/FAIR-2022


In [2]:
def svm_lib(X_train, y_train, X_test):
    svc = SVC(probability=True, kernel='linear')
    model = svc.fit(X_train, y_train)
    return model.predict(X_test)

def wsvm(C, X_train, y_train, X_test, distribution_weight=None):
    model = Wsvm(C, distribution_weight)
    model.fit(X_train, y_train)
    return model.predict(X_test)

def svm(C, X_train, y_train, X_test):
    model = Svm(C)
    model.fit(X_train, y_train)
    return model.predict(X_test)

In [3]:
def is_tomek(X, y, class_type):
    nn = NearestNeighbors(n_neighbors=2)
    nn.fit(X)
    _, neighbor_indices = nn.kneighbors(X)
    nn_index = neighbor_indices[:, 1]
    links = np.zeros(len(y), dtype=bool)
    class_excluded = [c for c in np.unique(y) if c not in class_type]
    X_dangxet, X_tl = [], []
    
    for index_sample, target_sample in enumerate(y):
        if target_sample in class_excluded:
            continue
        if y[nn_index[index_sample]] != target_sample:
            if nn_index[nn_index[index_sample]] == index_sample:
                X_tl.append(index_sample)
                X_dangxet.append(nn_index[index_sample])
                links[index_sample] = True
    return links, X_dangxet, X_tl

def Gmean(y_test, y_pred):
    # Ép confusion matrix luôn trả về ma trận 2x2 đúng chuẩn nhãn [-1, 1] 
    # (Để tránh lỗi khi mô hình dự đoán toàn bộ là 1 nhãn)
    cm_WSVM = confusion_matrix(y_test, y_pred, labels=[-1.0, 1.0])
    TN, FP, FN, TP = cm_WSVM.ravel()
    
    # Tính toán an toàn (tránh lỗi chia cho 0)
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    gmean = math.sqrt(sensitivity * specificity)
    
    return specificity, sensitivity, gmean

def metr_text(f, X_train, y_test, test_pred, sp, se, gmean):
    f.write(f"\nSo luong samples training: {len(X_train)} | testing: {len(y_test)}\n")
    f.write(f"SP: {sp:0.4f} | SE: {se:0.4f} | Gmean: {gmean:0.4f} | F1: {f1_score(y_test, test_pred):0.4f} | AUC: {roc_auc_score(y_test, test_pred):0.4f}\n")

In [4]:
def compute_weight(X, y, name_method="actual_hyper_lin", name_function="exp", beta=None, C=None, gamma=None, u=None, sigma=None):
    method = fuzzy.method()
    function = fuzzy.function()
    pos_index = np.where(y == 1)
    neg_index = np.where(y == -1)
    
    if name_method == "own_class_center":
        d = method.own_class_center(X, y)
    elif name_method == "estimated_hyper_lin":
        d = method.estimated_hyper_lin(X, y)
    elif name_method == "own_class_center_opposite":
        d = method.own_class_center_opposite(X, y)
    elif name_method == 'actual_hyper_lin':
        d = method.actual_hyper_lin(X, y, C=C, gamma=gamma)
    elif name_method == "distance_center_own_opposite_tam": # [Của AFW-CIL]
        d_own, d_opp, d_tam = method.distance_center_own_opposite_tam(X,y)
    
    if name_function == "lin":
        W = function.lin(d)
    elif name_function == "exp":
        W = function.exp(d, beta)
    elif name_function == "lin_center_own":
        W = function.lin_center_own(d, pos_index, neg_index)
    elif name_function == "func_own_opp_new": # [Của AFW-CIL]
        W = function.func_own_opp_new(d_own, d_opp, pos_index, neg_index, d_tam)
        
    r_pos = 1
    r_neg = len(pos_index)/len(neg_index)
    W = np.array(W)
    m = np.zeros(len(y))
    m[pos_index] = W[pos_index] * r_pos
    m[neg_index] = W[neg_index] * r_neg
    return m

def fuzzy_weight(beta_center, beta_estimate, beta_actual, X_train, y_train, namemethod, namefunction):
    # Hàm wrap gọi gọn lại cho thuật toán
    if namemethod == "distance_center_own_opposite_tam" and namefunction == "func_own_opp_new":
        return compute_weight(X_train, y_train, name_method=namemethod, name_function=namefunction)
    # Có thể mở rộng cho các hàm khác nếu cần
    return compute_weight(X_train, y_train, name_method=namemethod, name_function=namefunction)

In [5]:
def data_tomelinks(C, ind_posX, ind_negX, weight, X_test, y_test, X_train, y_train, K_neighbors, sigma_1, sigma_2, sigma_3, sigma_4):
    new_W = np.copy(weight)
    pos_index = np.where(y_train == 1)
    neg_index = np.where(y_train == -1)
    
    clf = Wsvm(C, new_W)
    clf.fit(X_train, y_train)
    y_predict = clf.predict(X_test)
    specificity, sensitivity, gmean = Gmean(y_test, y_predict)
    
    nn2 = NearestNeighbors(n_neighbors=K_neighbors)
    nn2.fit(X_train)
    y_nn = []
    
    # [MODIFIED 1]: Phạt nhiễu âm (TH2) - Thay /1.2 bằng * sigma_2
    neg_pred = clf.predict(X_train[neg_index[0]])
    idx_neg_wrong = np.where(neg_pred != -1.0)[0]
    new_W[neg_index[0][idx_neg_wrong]] = new_W[neg_index[0][idx_neg_wrong]] * sigma_2 

    ind_nn = []
    for ind, i in enumerate(ind_posX):
        sample_i = X_train[i:i+1]
        y_pred = clf.predict(sample_i)
        
        if y_pred == -1.0: # Ranh giới lấn sang âm (TH3)
            ind_nn.append(ind)
            knn_indices = nn2.kneighbors(sample_i)[1].flatten()
            for j in knn_indices:
                y_nn.append(y_train[j]) 
        else: # Ranh giới lấn sang dương (TH1) - Thay *1.2 và /1.2 bằng sigma_1
            new_W[ind_negX[ind]] = new_W[ind_negX[ind]] * (1 - sigma_1)
            new_W[i] = new_W[i] * (1 + sigma_1)
            
    ind_nn = np.array(ind_nn)
    y_nn = np.array(y_nn)
    if len(y_nn) > 0:
        y_nn = np.array_split(y_nn, len(y_nn) / K_neighbors)
        for ind, i in enumerate(range(0, len(y_nn))):   
            if 1 not in y_nn[i][1:]: # Phạt nhiễu dương (TH4) - Thay /1.2 bằng * sigma_4
                new_W[ind_posX[ind_nn[ind]]] = new_W[ind_posX[ind_nn[ind]]] * sigma_4  # FIX: bỏ [0] vì ind_posX là list int
            else: # Tiếp tục xử lý lấn ranh giới âm (TH3) - Thay bằng sigma_3
                new_W[ind_negX[ind_nn[ind]]] = new_W[ind_negX[ind_nn[ind]]] * (1 - sigma_3)  # FIX: bỏ [0]
                new_W[ind_posX[ind_nn[ind]]] = new_W[ind_posX[ind_nn[ind]]] * (1 + sigma_3)  # FIX: bỏ [0]

    return new_W, gmean, sensitivity


In [6]:
def lfb(C, ind_posX, ind_negX, weight, namemethod, namefunction, T, X_test, y_test, X_train, y_train, K_neighbors, s1, s2, s3, s4):
    gmax = 0
    best_weight = np.copy(weight)
    current_weight = np.copy(weight)
    
    for i in range(0, T):
        current_weight, gmean, se = data_tomelinks(
            C, ind_posX, ind_negX, current_weight, X_test, y_test, X_train, y_train, 
            K_neighbors, s1, s2, s3, s4
        )
        if gmean > gmax:
            gmax = gmean
            best_weight = np.copy(current_weight)
            
    return best_weight, gmax

In [7]:
# Thuật toán PSO tự viết bằng Numpy (Đã fix lỗi Tuple bounds)
class PSO_AFWCIL:
    def __init__(self, num_particles, iters, bounds, C, T, X_train, y_train, X_test, y_test, namemethod, namefunction):
        self.num_particles = num_particles
        self.iters = iters
        self.bounds = bounds # bounds = [(K_min, K_max), (s1_min, s1_max), ...]
        self.C = C
        self.T = T
        self.X_train, self.y_train = X_train, y_train
        self.X_test, self.y_test = X_test, y_test
        self.namemethod = namemethod
        self.namefunction = namefunction
        
        # Tiền xử lý 1 lần cho toàn bộ PSO
        self.links, self.ind_posX, self.ind_negX = is_tomek(X_train, y_train, class_type=[-1.0])
        self.init_weight = fuzzy_weight(0.5, 0.8, 0.2, X_train, y_train, namemethod, namefunction)
        
    def fitness_function(self, particle):
        # [FIX] Bóc tách đúng index của từng tham số từ hạt (particle)
        K = int(round(particle[0]))  # FIX: particle[0] thay vì particle (tránh lỗi round(tuple))
        s1, s2, s3, s4 = particle[1], particle[2], particle[3], particle[4]
        
        _, gmax = lfb(self.C, self.ind_posX, self.ind_negX, self.init_weight, 
                      self.namemethod, self.namefunction, self.T, 
                      self.X_test, self.y_test, self.X_train, self.y_train, 
                      K, s1, s2, s3, s4)
        return gmax

    def optimize(self):
        dim = len(self.bounds)
        # Khởi tạo vị trí ngẫu nhiên
        pos = np.random.rand(self.num_particles, dim)
        
        # [FIX] Cập nhật mảng đúng cách (Max - Min) + Min
        for i in range(dim):
            min_val = self.bounds[i][0]  # FIX: self.bounds[i][0] thay vì self.bounds[i]
            max_val = self.bounds[i][1]
            pos[:, i] = pos[:, i] * (max_val - min_val) + min_val
        
        vel = np.random.randn(self.num_particles, dim) * 0.1
        
        pbest_pos = np.copy(pos)
        pbest_val = np.zeros(self.num_particles) - 1.0 # Khởi tạo fitness rất thấp
        gbest_pos = np.zeros(dim)
        gbest_val = -1.0
        
        # Vòng lặp PSO
        for i in range(self.iters):
            for j in range(self.num_particles):
                fit_val = self.fitness_function(pos[j])
                
                # Cập nhật pBest và gBest
                if fit_val > pbest_val[j]:
                    pbest_val[j] = fit_val
                    pbest_pos[j] = np.copy(pos[j])
                if fit_val > gbest_val:
                    gbest_val = fit_val
                    gbest_pos = np.copy(pos[j])
            
            # Cập nhật vận tốc và vị trí (công thức PSO chuẩn)
            r1, r2 = np.random.rand(), np.random.rand()
            vel = 0.5 * vel + 1.5 * r1 * (pbest_pos - pos) + 1.5 * r2 * (gbest_pos - pos)
            pos = pos + vel
            
            # [FIX] Ràng buộc không gian (Clipping) đúng cách bằng Tuple bounds
            for d in range(dim):
                pos[:, d] = np.clip(pos[:, d], self.bounds[d][0], self.bounds[d][1])  # FIX: self.bounds[d][0]
                
            print(f"Iteration {i+1}/{self.iters} | Best G-mean: {gbest_val:.4f} | Params: K={int(round(gbest_pos[0]))}, s1={gbest_pos[1]:.2f}, s2={gbest_pos[2]:.2f}, s3={gbest_pos[3]:.2f}, s4={gbest_pos[4]:.2f}")
            
        return gbest_pos, gbest_val

In [8]:
import time as timer
import pandas as pd
from sklearn.model_selection import train_test_split as tts
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ============================================================
# 0. CHỌN DATASET (Bật 1 trong các dòng dưới, tắt các dòng còn lại)
# ============================================================
DATASET_NAME = "haberman"        # << THAY TÊN DATASET TẠI ĐÂY
# Các giá trị hợp lệ:
#   "haberman"       -> haberman.csv          (imbalance ~ 1:3)
#   "vertebral"      -> Vertebral_column.csv  (imbalance ~ 1:3)
#   "ilpd"           -> ILPD.csv              (indian liver patient)
#   "german_credit"  -> german_credit.csv
#   "abalone19"      -> abalone19.csv         (imbalance ~ 1:9)
#   "spect_heart"    -> Spect_Heart.csv
#   "transfusion"    -> transfusion.csv
#   "glass1"         -> glass1.csv
#   "pima"           -> diabetes.csv          (Pima Indians Diabetes)
#   "yeast"          -> yeast.csv

TEST_SIZE = 0.25       # Tỷ lệ test
NEW_RATE  = None       # None = dùng tỷ lệ gốc; số thực (vd. 1/9) = resample mất cân bằng
N_PCA     = None       # None = không dùng PCA; số nguyên = số thành phần PCA

# ============================================================
# 1. HÀM LOAD DỮ LIỆU TỔNG QUÁT (Tự nhận dataset qua DATASET_NAME)
# ============================================================
DATASET_ROOT = os.path.join(os.path.dirname(os.getcwd()), 'FAIR-2022', 'Processing_Data', 'dataset')
# Hoặc dùng đường dẫn tuyệt đối nếu cần:
# DATASET_ROOT = '../Processing_Data/dataset'

DATASET_CONFIG = {
    "haberman": {
        "file": "haberman.csv",
        "label_col": "class",
        "label_map": {2: 1.0, 1: -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "vertebral": {
        "file": "Vertebral_column.csv",  # trong data/datasets/
        "label_col": "Label class",
        "label_map": {"Abnormal": -1.0, "Normal": 1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "ilpd": {
        "file": "Indian-Liver-Patient-Dataset-ILPD.csv",
        "label_col": "Dataset",
        "label_map": {1: -1.0, 2: 1.0},
        "drop_cols": [],
        "cat_cols": ["Gender"],          # sẽ OneHot encode
        "impute": True,
        "pca_default": None,
    },
    "german_credit": {
        "file": "german_credit.csv",
        "label_col": "default",
        "label_map": {1: -1.0, 0: 1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "abalone19": {
        "file": "abalone19.csv",
        "label_col": "Class_abalone19",
        "label_map": {"positive": 1.0, "negative": -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "spect_heart": {
        "file": "Spect_Heart.csv",
        "label_col": "OVERALL_DIAGNOSIS",
        "label_map": {1: 1.0, 0: -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "transfusion": {
        "file": "transfusion.csv",
        "label_col": "whether he/she donated blood in March 2007",
        "label_map": {1: 1.0, 0: -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "glass1": {
        "file": "glass1.csv",
        "label_col": "Class",
        "label_map": {"positive": 1.0, "negative": -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "pima": {
        "file": "diabetes.csv",
        "label_col": "Outcome",
        "label_map": {1: 1.0, 0: -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
    "yeast": {
        "file": "yeast.csv",
        "label_col": "Class",
        "label_map": {"positive": 1.0, "negative": -1.0},
        "drop_cols": [],
        "cat_cols": [],
        "impute": False,
        "pca_default": None,
    },
}

def load_real_data(name, test_size=0.25, new_rate=None, n_pca=None):
    """
    Load và tiền xử lý bất kỳ dataset nào đã cấu hình trong DATASET_CONFIG.
    Trả về: X_train, y_train, X_test, y_test (nhãn -1 và 1, chuẩn SVM)
    """
    cfg = DATASET_CONFIG[name]
    
    # --- Tìm file (thử trong Processing_Data/dataset trước, rồi data/datasets) ---
    search_dirs = [
        os.path.join('../Processing_Data/dataset'),
        os.path.join('../data/datasets'),
        os.path.join('../data'),
    ]
    fpath = None
    for d in search_dirs:
        candidate = os.path.join(d, cfg["file"])
        if os.path.exists(candidate):
            fpath = candidate
            break
    if fpath is None:
        raise FileNotFoundError(f"Không tìm thấy file '{cfg['file']}' trong các thư mục dataset.")
    
    df = pd.read_csv(fpath)
    print(f"[✔] Đã load: {fpath}  | Shape: {df.shape}")
    
    # --- Ánh xạ nhãn ---
    lbl_col = cfg["label_col"]
    df[lbl_col] = df[lbl_col].map(cfg["label_map"])
    df = df.dropna(subset=[lbl_col])  # Xoá dòng nhãn không map được
    
    y = df[lbl_col].values.astype(float)
    X_df = df.drop(columns=[lbl_col] + cfg.get("drop_cols", []))
    
    # --- OneHot cho cột phân loại ---
    if cfg["cat_cols"]:
        cat_indices = [X_df.columns.get_loc(c) for c in cfg["cat_cols"]]
        ct = ColumnTransformer(
            [('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_indices)],
            remainder='passthrough'
        )
        X = ct.fit_transform(X_df.values)
    else:
        X = X_df.values.astype(float)
    
    # --- Impute NaN ---
    if cfg["impute"]:
        imp = SimpleImputer(strategy='mean')
        X = imp.fit_transform(X)
    
    # --- Resample tỷ lệ mất cân bằng (tuỳ chọn) ---
    if new_rate is not None:
        from data.common.change_rate_data import change_rate_data
        X, y = change_rate_data(X, y, new_rate=new_rate)
    
    # --- Chia tập Train/Test ---
    X_train, X_test, y_train, y_test = tts(X, y, test_size=test_size, random_state=42, stratify=y)
    
    # --- Chuẩn hoá ---
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test  = sc.transform(X_test)
    
    # --- PCA (tuỳ chọn) ---
    n_components = n_pca if n_pca is not None else cfg.get("pca_default")
    if n_components is not None:
        pca = PCA(n_components=n_components)
        X_train = pca.fit_transform(X_train)
        X_test  = pca.transform(X_test)
    
    y_train = np.array(y_train)
    y_test  = np.array(y_test)
    return X_train, y_train, X_test, y_test

# ============================================================
# 2. NẠP DỮ LIỆU
# ============================================================
print(f"Đang nạp dataset: [{DATASET_NAME}] ...")
X_train, y_train, X_test, y_test = load_real_data(DATASET_NAME, test_size=TEST_SIZE, new_rate=NEW_RATE, n_pca=N_PCA)

pos_train = np.sum(y_train == 1)
neg_train = np.sum(y_train == -1)
print(f"Train -> Shape: {X_train.shape} | Dương (+1): {pos_train} | Âm (-1): {neg_train} | IR: {neg_train/pos_train:.2f}")
print(f"Test  -> Shape: {X_test.shape}  | Dương (+1): {np.sum(y_test==1)} | Âm (-1): {np.sum(y_test==-1)}")

# ============================================================
# 3. CẤU HÌNH THUẬT TOÁN
# ============================================================
C = 100
T = 5          # Số vòng lặp AFW-CIL
namemethod = "distance_center_own_opposite_tam"
namefunction = "func_own_opp_new"

# ============================================================
# 4. BASELINE W-SVM (Không xử lý mờ, không SMOTE)
# ============================================================
print("\n" + "="*60)
print("=== 1. BASELINE W-SVM ===")
w_base = np.ones(len(y_train))
pred_wsvm = wsvm(C, X_train, y_train, X_test, w_base)
sp_b, se_b, gm_b = Gmean(y_test, pred_wsvm)
print(f"SP: {sp_b:.4f} | SE: {se_b:.4f} | G-mean: {gm_b:.4f} | AUC: {roc_auc_score(y_test, pred_wsvm):.4f}")

# ============================================================
# 5. BORDERLINE-SMOTE + W-SVM
# ============================================================
print("\n" + "="*60)
print("=== 2. BORDERLINE-SMOTE + W-SVM ===")
smote = BorderlineSMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)[:2]
w_smote = np.ones(len(y_train_sm))
pred_smote = wsvm(C, X_train_sm, y_train_sm, X_test, w_smote)
sp_sm, se_sm, gm_sm = Gmean(y_test, pred_smote)
print(f"SP: {sp_sm:.4f} | SE: {se_sm:.4f} | G-mean: {gm_sm:.4f} | AUC: {roc_auc_score(y_test, pred_smote):.4f}")

# ============================================================
# 6. ĐỀ XUẤT: PSO-AFWCIL TRÊN DATA GỐC
# ============================================================
print("\n" + "="*60)
print("=== 3. PSO-AFWCIL (Data gốc, không SMOTE) ===")
bounds = [(3, 11), (0.01, 0.5), (0.01, 0.99), (0.01, 0.5), (0.01, 0.99)]
t0 = timer.time()
pso_orig = PSO_AFWCIL(num_particles=5, iters=5, bounds=bounds, C=C, T=T,
                      X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
                      namemethod=namemethod, namefunction=namefunction)
best_params_orig, best_gmean_orig = pso_orig.optimize()
print(f"[PSO-AFWCIL gốc] G-mean tốt nhất: {best_gmean_orig:.4f} | Thời gian: {timer.time()-t0:.1f}s")
print(f"Tham số tốt nhất: K={int(round(best_params_orig[0]))}, s1={best_params_orig[1]:.3f}, s2={best_params_orig[2]:.3f}, s3={best_params_orig[3]:.3f}, s4={best_params_orig[4]:.3f}")

# ============================================================
# 7. ĐỀ XUẤT: SMOTE + PSO-AFWCIL (Hybrid)
# ============================================================
print("\n" + "="*60)
print("=== 4. HYBRID: SMOTE + PSO-AFWCIL ===")
t1 = timer.time()
pso_smote = PSO_AFWCIL(num_particles=5, iters=5, bounds=bounds, C=C, T=T,
                       X_train=X_train_sm, y_train=y_train_sm, X_test=X_test, y_test=y_test,
                       namemethod=namemethod, namefunction=namefunction)
best_params_sm, best_gmean_sm = pso_smote.optimize()
print(f"[SMOTE+PSO-AFWCIL] G-mean tốt nhất: {best_gmean_sm:.4f} | Thời gian: {timer.time()-t1:.1f}s")
print(f"Tham số tốt nhất: K={int(round(best_params_sm[0]))}, s1={best_params_sm[1]:.3f}, s2={best_params_sm[2]:.3f}, s3={best_params_sm[3]:.3f}, s4={best_params_sm[4]:.3f}")

# ============================================================
# 8. TỔNG KẾT
# ============================================================
print("\n" + "="*60)
print(f"{'Phương pháp':<30} {'G-mean':>8}")
print("-"*40)
print(f"{'W-SVM (Baseline)':<30} {gm_b:>8.4f}")
print(f"{'SMOTE + W-SVM':<30} {gm_sm:>8.4f}")
print(f"{'PSO-AFWCIL (gốc)':<30} {best_gmean_orig:>8.4f}")
print(f"{'SMOTE + PSO-AFWCIL (Hybrid)':<30} {best_gmean_sm:>8.4f}")


Đang nạp dataset: [haberman] ...
[✔] Đã load: /home/quangvd/project/FAIR-2022/Processing_Data/dataset/haberman.csv  | Shape: (306, 4)
Train -> Shape: (229, 3) | Dương (+1): 61 | Âm (-1): 168 | IR: 2.75
Test  -> Shape: (77, 3)  | Dương (+1): 20 | Âm (-1): 57

=== 1. BASELINE W-SVM ===
SP: 0.8772 | SE: 0.2000 | G-mean: 0.4189 | AUC: 0.5386

=== 2. BORDERLINE-SMOTE + W-SVM ===
SP: 0.8070 | SE: 0.4000 | G-mean: 0.5682 | AUC: 0.6035

=== 3. PSO-AFWCIL (Data gốc, không SMOTE) ===
Iteration 1/5 | Best G-mean: 0.5682 | Params: K=10, s1=0.18, s2=0.06, s3=0.40, s4=0.78
Iteration 2/5 | Best G-mean: 0.6026 | Params: K=10, s1=0.22, s2=0.01, s3=0.38, s4=0.87
Iteration 3/5 | Best G-mean: 0.6026 | Params: K=10, s1=0.22, s2=0.01, s3=0.38, s4=0.87
Iteration 4/5 | Best G-mean: 0.6026 | Params: K=10, s1=0.22, s2=0.01, s3=0.38, s4=0.87
Iteration 5/5 | Best G-mean: 0.6026 | Params: K=10, s1=0.22, s2=0.01, s3=0.38, s4=0.87
[PSO-AFWCIL gốc] G-mean tốt nhất: 0.6026 | Thời gian: 3.4s
Tham số tốt nhất: K=10, s1=